---
icon: lucide/ruler
---

# Physical Units

<em>k</em>UPS stores positions, energies, and masses as JAX arrays, and a JAX array only stores numbers.
The unit is a convention: Å for lengths, eV for energies, amu for masses, $e$ for charges, and K for temperatures.
Nothing checks this convention, so a force field parametrized in kcal/mol, a pressure given in bar, or a missing factor of $k_B$ gives numbers that look plausible and are wrong.

[kups.core.units][kups.core.units] makes the convention explicit.
A *quantity* ([Quantity][kups.core.units.Quantity]) is an array together with its physical dimension.
Dimensions are checked twice: (1) at runtime, when JAX traces a function, so a mismatch raises before any computation runs, and (2) statically, when [pyrefly](https://pyrefly.org) checks the code.
Since the dimension is pytree metadata and not data, a compiled function is the same program as one written with plain arrays.

This notebook covers (1) quantities and the base system that stores them, (2) units from text and physical constants, (3) gradients, JAX transformations, and array functions, (4) static and configuration checks, and (5) the pitfalls.

In [ ]:
import jax
import jax.numpy as jnp

from kups.core import units
from kups.core.units import catalog as u
from kups.core.units import constants as const

jax.config.update("jax_enable_x64", True)

## Quantities

Multiplying a number or an array by a unit gives a quantity.
Arithmetic propagates the dimension, and `.to(unit)` returns the plain magnitudes in any compatible unit.
Adding 0.5 kcal/mol to 0.01 eV gives 0.0317 eV, with the conversion done for you.
Adding incompatible quantities raises a [DimensionError][kups.core.units.DimensionError] that names both dimensions.

In [ ]:
r = jnp.array([3.4, 3.8, 4.5]) * u.angstrom
print(r)
print(r.to(u.nm))

energy = 0.5 * u.kcal / u.mol + 0.01 * u.eV
print(energy)

In [ ]:
try:
    r + 1.0 * u.eV
except units.DimensionError as err:
    print(err)

## Base Systems

A quantity stores its magnitudes in a *base system*, i.e., a set of reference units from which every other unit is derived.
The default, `ATOMISTIC`, uses Å, amu, eV, $e$, and K.
`q.value` is the magnitude in these units, so moving between a quantity and a plain <em>k</em>UPS array never introduces a conversion factor.
Derived dimensions use the same base.
The base time unit, for instance, is $\text{Å}\sqrt{\text{amu}/\text{eV}} \approx 10.18$ fs, so a time step of 0.5 fs has the magnitude 0.0491.
`units.base` switches the base inside a `with` block.

In [ ]:
dt = 0.5 * u.fs
print(dt.value)  # magnitude in the base time unit
print(dt.to(u.fs))

with units.base(units.SI):
    print(3.0 * u.angstrom)

**Why not SI.** float32 keeps the same relative accuracy at any magnitude, but its range ends near $10^{-38}$.
SI magnitudes at the atomic scale sit close to that edge: 1 amu is $1.66 \times 10^{-27}$ kg, and its square, $2.8 \times 10^{-54}$ kg², underflows to zero in float32.
In the atomistic base, typical simulation values are of order one.
The price is that derived units of the base are unusual, such as the 10.18 fs time unit, and every conversion goes through the base.
Converting 0.5 fs there and back gives 0.49999999999999994, one unit in the last place off.

## Units from Text

Configuration files store quantities as strings.
[parse][kups.core.units.catalog.parse] reads a number followed by a unit expression, e.g., `"1 kcal/mol/angstrom"`.
Every unit without a prefix accepts the SI prefixes (`kJ`, `fs`, `GPa`), and a scalar quantity prints as text `parse` reads back.

In [ ]:
print(u.parse("1 bar").to(u.eV / u.angstrom**3))
print(u.parse("1 kcal/mol/angstrom").to(u.eV / u.angstrom))
print(u.parse("120 kJ/mol").to(u.eV))

[constants][kups.core.units.constants] provides the CODATA 2018 values as quantities.
A forgotten $k_B$ therefore raises, since a temperature cannot be added to an energy.

In [ ]:
kT = const.k_B * (300.0 * u.K)
print(kT)
print(const.k_e)  # Coulomb constant

## Gradients

Forces are negative gradients of the energy with respect to the positions, so their dimension is energy per length.
[units.grad][kups.core.units.grad] gives the derivative the dimension $\dim(\text{out}) / \dim(\text{arg})$ and `value_and_grad` follows the same rule. `vjp` also multiplies in the dimension of the cotangent it pulls back, so a plain cotangent gives the same labels as `grad`. `hessian` divides by the argument's dimension twice.
Below, the Lennard-Jones forces at three argon pair distances come out in eV/Å.

In [ ]:
def lennard_jones(r, epsilon, sigma):
    s6 = (sigma / r) ** 6
    return 4 * epsilon * (s6**2 - s6)


epsilon, sigma = 0.0104 * u.eV, 3.4 * u.angstrom


def energy(r):
    return lennard_jones(r, epsilon, sigma).sum()


forces = -units.grad(energy)(r)
print(forces)

`jax.grad` cannot follow this rule.
It rejects a quantity as the output, but once the energy is converted to a plain array, it returns a gradient with the dimension of `r`.
The numbers are the right ones in eV/Å, the label Å is wrong, and nothing raises:

In [ ]:
print(jax.grad(lambda r: energy(r).to(u.eV))(r))  # wrong: labelled Å

## JAX Transformations

The magnitudes are the only leaf of a quantity's pytree, and the dimension is part of the tree structure.
`jax.jit`, `jax.vmap`, and `jax.lax.scan` therefore accept quantities unchanged, and two quantities share a tree structure exactly when their dimensions and bases agree.

**The price.** Unit conversions bake the scale factors of the active base into the traced program.
The active base is therefore part of the `jax.jit` cache key, and switching it with `units.base` retraces every jitted function. Like JAX's own config contexts, a `units.base` scope holds for the thread that enters it.

Apart from these constants, the traced program is the one plain arrays give.
The comparison of the two jaxprs below prints `True`:

In [ ]:
pairwise = jax.jit(jax.vmap(lambda r: lennard_jones(r, epsilon, sigma)))
print(pairwise(r))

with_units = jax.make_jaxpr(lambda a: (a * a).sum())(r)
without_units = jax.make_jaxpr(lambda a: (a * a).sum())(r.value)
print(str(with_units) == str(without_units))

## Arrays and Linear Algebra

Quantities support the usual array methods: `sum`, `mean`, `reshape`, indexing, and `.at[]`.
`jax.numpy` rejects quantities, since most of its functions cannot know the dimension of their result.
<em>k</em>UPS provides unit-aware versions where the rule is not obvious, e.g., [norm][kups.core.units.norm], [where][kups.core.units.where], [einsum][kups.core.units.einsum], and [eigh][kups.core.units.eigh].
For a Hessian in eV/Å², the eigenvalues carry its dimension and the eigenvectors are plain arrays.

In [ ]:
hessian = jnp.array([[2.0, 0.5], [0.5, 1.0]]) * u.eV / u.angstrom**2
eigenvalues, eigenvectors = units.eigh(hessian)
print(eigenvalues)
print(type(eigenvectors).__name__)

Any other function that keeps the dimension of its input, such as `jnp.cumsum`, can be lifted with [apply][kups.core.units.apply].
[relabel][kups.core.units.relabel] states a dimension that cannot be inferred.

In [ ]:
print(units.apply(jnp.cumsum, r))
print(units.relabel(jnp.ones(2), units.dim_of(u.eV)))

## Static Checking

[Quantity][kups.core.units.Quantity] and [Unit][kups.core.units.Unit] are generic over the five exponents of their dimension in $(\text{m}, \text{kg}, \text{s}, \text{A}, \text{K})$.
An energy, for instance, is `Quantity[2, 1, -2, 0, 0]`, and aliases such as `Energy` and `Velocity` name the common cases.
pyrefly does arithmetic on these exponents, so it checks the function below without running it.
Had the body returned `0.5 * mass * velocity`, pyrefly would report

```
Returned type `Quantity[1, 1, -1, 0, 0]` is not assignable to declared return type `Quantity[2, 1, -2, 0, 0]`
```

Two limits apply: (1) only pyrefly checks the exponents, while other type checkers accept the code without checking dimensions, and (2) the square root of a dimension with an odd exponent, such as a length, has no integer-exponent type.
pyrefly types it as `OddExponentRoot`, which supports no operations; the runtime check still covers it.

In [ ]:
def kinetic_energy(mass: units.Mass, velocity: units.Velocity) -> units.Energy:
    return 0.5 * mass * velocity**2


mass = 39.95 * u.amu  # argon
velocity = units.sqrt(kT / mass)
print(kinetic_energy(mass, velocity))

## Validated Configuration

The dimension aliases also work as [pydantic](https://docs.pydantic.dev) field types.
The annotation fixes the dimension, and the input chooses the unit.
Compatible values are converted, and incompatible ones are rejected with the expected and received dimensions.
Below, a temperature given in eV fails validation.

In [ ]:
from pydantic import BaseModel, ValidationError


class MDConfig(BaseModel):
    temperature: units.Temperature
    pressure: units.Pressure
    time_step: units.Time


config = MDConfig.model_validate(
    {"temperature": "300 K", "pressure": "1 bar", "time_step": "0.5 fs"}
)
print(config.time_step.to(u.fs))

try:
    MDConfig.model_validate(
        {"temperature": "300 eV", "pressure": "1 bar", "time_step": "0.5 fs"}
    )
except ValidationError as err:
    print(err.errors()[0]["msg"])

## Pitfalls

A quantity never converts to an array implicitly.
`jnp.asarray(q)` and `q == q` raise, since both would silently drop the dimension.
Use `q.to(unit)` or `q.value` to get magnitudes, and [testing.assert_allclose][kups.core.units.testing.assert_allclose] to compare quantities in a named unit.

A plain number never stands in for a dimensioned operand, zero included.
`q + 0`, `q > 0`, and `q.at[i].set(0.0)` all raise, so write the zero with its unit, `0.0 * u.eV`, or use `units.zeros_like(q)`.

In [ ]:
from kups.core.units import testing

testing.assert_allclose(1.0 * u.nm, 10.0 * u.angstrom, u.angstrom)

`jax.tree.map` over a single tree treats a quantity as a node and maps over its magnitudes.
A nonlinear function therefore keeps the old unit: the square root of 4 Å² prints as 2 Å².
Use [units.tree_map][kups.core.units.tree_map] instead, or pass `is_leaf=units.is_leaf`.

In [ ]:
area = 4.0 * u.angstrom**2
print(jax.tree.map(jnp.sqrt, area))  # wrong: labelled Å^2
print(units.tree_map(units.sqrt, area))